# Inicialización y estabilidad

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_multilayer-perceptrons/numerical-stability-and-init.ipynb` · [Lección original](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Estabilidad numérica e inicialización
<a id="sec_numerical_stability"></a>

Todos los modelos anteriores empezaban con parámetros inicializados según alguna distribución. Hasta ahora aceptábamos esa elección sin detenernos en sus consecuencias. Sin embargo, la inicialización influye en la velocidad de aprendizaje y puede ser decisiva para mantener activaciones y gradientes en una escala útil.

La elección está ligada a la activación: un esquema apropiado para una red con tanh no tiene por qué ser el más adecuado para ReLU. Una mala combinación puede producir saturación, gradientes que se desvanecen o valores que crecen demasiado. En esta sección examinaremos esos mecanismos y las reglas de inicialización que ayudan a controlarlos.


In [ ]:
%matplotlib inline
import torch
from laboratorio import d2l

## Desvanecimiento y explosión de gradientes
Considere una red profunda con capas $L$, entrada $\mathbf{x}$ y salida $\mathbf{o}$. Con cada capa $l$ definida por una transformación $f_l$ parametrizada por pesos $\mathbf{W}^{(l)}$, cuya salida oculta de capa es $\mathbf{h}^{(l)}$ (sea $\mathbf{h}^{(0)} = \mathbf{x}$), nuestra red se puede expresar como:

$$\mathbf{h}^{(l)} = f_l (\mathbf{h}^{(l-1)}) \textrm{ and thus } \mathbf{o} = f_L \circ \cdots \circ f_1(\mathbf{x}).$$

Si toda la salida de capa oculta y la entrada son vectores, podemos escribir el gradiente de $\mathbf{o}$ con respecto a cualquier conjunto de parámetros $\mathbf{W}^{(l)}$ de la siguiente manera:

$$\partial_{\mathbf{W}^{(l)}} \mathbf{o} = \underbrace{\partial_{\mathbf{h}^{(L-1)}} \mathbf{h}^{(L)}}_{ \mathbf{M}^{(L)} \stackrel{\textrm{def}}{=}} \cdots \underbrace{\partial_{\mathbf{h}^{(l)}} \mathbf{h}^{(l+1)}}_{ \mathbf{M}^{(l+1)} \stackrel{\textrm{def}}{=}} \underbrace{\partial_{\mathbf{W}^{(l)}} \mathbf{h}^{(l)}}_{ \mathbf{v}^{(l)} \stackrel{\textrm{def}}{=}}.$$

En otras palabras, este gradiente es el producto de $L-l$ matrices $\mathbf{M}^{(L)} \cdots \mathbf{M}^{(l+1)}$ y el vector de gradiente $\mathbf{v}^{(l)}$. Así que somos susceptibles a los mismos problemas de subflujo numérico que a menudo surgen cuando se multiplican demasiadas probabilidades. Al tratar con probabilidades, un truco común es cambiar al espacio logarítmico, es decir, cambiar la presión de la mantisa al exponente de la representación numérica. Desafortunadamente, nuestro problema de arriba es más grave: inicialmente las matrices $\mathbf{M}^{(l)}$ pueden tener una amplia variedad de valores propios. Pueden ser pequeños o grandes, y su producto puede ser *muy grande* o *muy pequeño*.

Los riesgos que plantean los gradientes inestables van más allá de la representación numérica. Los gradientes de magnitud impredecible también amenazan la estabilidad de nuestros algoritmos de optimización. Podemos estar enfrentando actualizaciones de parámetros que son (i) excesivamente grandes, destruyendo nuestro modelo (el problema de explosión de gradientes); o (ii) excesivamente pequeños (el problema de desvanecimiento de gradientes), haciendo imposible el aprendizaje mientras los parámetros apenas se mueven en cada actualización.

### Desvanecimiento de gradientes

Un culpable frecuente que causa el problema del desvanecimiento de gradientes es la elección de la función de activación $\sigma$ que se adjunta después de las operaciones lineales de cada capa. Históricamente, la función sigmoide $1/(1 + \exp(-x))$ (introducida en [Referencia sec_mlp](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#sec-mlp)) era popular porque se asemeja a una función de umbral. Puesto que las primeras redes neuronales artificiales fueron inspiradas por redes neuronales biológicas, la idea de neuronas que disparan *completamente* o *no en absoluto* (como neuronas biológicas) parecía atractiva. Echemos un vistazo más de cerca al sigmoide para ver por qué puede causar gradientes de desaparición.


In [ ]:
x = torch.arange(-8.0, 8.0, 0.1, requires_grad=True)
y = torch.sigmoid(x)
y.backward(torch.ones_like(x))

d2l.plot(x.detach().numpy(), [y.detach().numpy(), x.grad.numpy()],
         legend=['sigmoid', 'gradient'], figsize=(4.5, 2.5))

### Nota docente de Hespérides

Un diagnóstico práctico comienza con un único minibatch: verifica formas, etiquetas, finitud de la pérdida y gradientes; comprueba después si puedes sobreajustarlo. Si no puedes, añadir regularización raramente resuelve la causa. La inicialización trata de controlar la escala de señales y gradientes. Gradient clipping limita la norma de un gradiente ya calculado, pero no repara un `NaN` ni un grafo desconectado. Registra las normas antes y después del recorte.

Vínculo con los apuntes: sesión 3, «Inicialización y estabilidad».


Como puedes ver, ** el gradiente del sigmoide desaparece tanto cuando sus entradas son grandes como cuando son pequeñas**. Además, cuando se retropropaga a través de muchas capas, a menos que estemos en la zona de Ricitos de Oro, donde las entradas de muchos de los sigmoides están cerca de cero, los gradientes del producto en general pueden desaparecer. Cuando nuestra red cuenta con muchas capas, a menos que tengamos cuidado, el gradiente probablemente será cortado en alguna capa. De hecho, este problema utilizado para plagar el entrenamiento de la red profunda. En consecuencia, ReLUs, que son más estables (pero menos neurológicamente plausible), han surgido como la opción por defecto para los practicantes.

### Explosión de gradientes

El problema opuesto, cuando los gradientes explotan, puede ser igualmente molesto. Para ilustrar esto un poco mejor, dibujamos 100 matrices aleatorias gaussianas y las multiplicamos con una matriz inicial. Para la escala que elegimos (la elección de la varianza $\sigma^2=1$), el producto de la matriz explota. Cuando esto sucede debido a la inicialización de una red profunda, no tenemos ninguna posibilidad de conseguir un optimizador de descenso por gradiente para converger.


In [ ]:
M = torch.normal(0, 1, size=(4, 4))
print('a single matrix \n',M)
for i in range(100):
    M = M @ torch.normal(0, 1, size=(4, 4))
print('after multiplying 100 matrices\n', M)

### Rompiendo la simetría
Otro problema en el diseño de redes neuronales es la simetría inherente a su parametrización. Supongamos que tenemos un simple MLP con una capa oculta y dos unidades. En este caso, podríamos permutar los pesos $\mathbf{W}^{(1)}$ de la primera capa y también permutar los pesos de la capa de salida para obtener la misma función. No hay nada especial diferenciando las unidades ocultas primera y segunda. En otras palabras, tenemos simetría de permutación entre las unidades ocultas de cada capa.

Esto es más que una molestia teórica. Consideremos el MLP de una capa oculta con dos unidades ocultas. Para ilustrar, supongamos que la capa de salida transforma las dos unidades ocultas en una sola unidad de salida. Imaginemos lo que sucedería si inicializáramos todos los parámetros de la capa oculta como $\mathbf{W}^{(1)} = c$ para alguna constante $c$. En este caso, durante la propagación hacia delante o unidad oculta toma las mismas entradas y parámetros que producen la misma activación que se alimenta a la unidad de salida. Durante la retropropagación, diferenciando la unidad de salida con respecto a los parámetros $\mathbf{W}^{(1)}$ da un gradiente todos cuyos elementos toman el mismo valor. Así, después de la iteración basada en el gradiente (por ejemplo, descenso por gradiente estocástico minibatch), todos los elementos de $\mathbf{W}^{(1)}$ todavía toman el mismo valor. Tales iteraciones nunca romperían la simetría* por sí solas y podríamos nunca ser capaces de realizar la potencia expresiva de la red. La capa oculta se comportaría como si tuviera sólo una unidad.

## Inicialización del parámetro
Una manera de abordar ---o al menos mitigar--- los problemas planteados anteriormente es a través de una cuidadosa inicialización. Como veremos más adelante, el cuidado adicional durante la optimización y regularización adecuada puede mejorar aún más la estabilidad.

### Inicialización predeterminada
En las secciones anteriores, por ejemplo, en [Referencia sec_linear_concise](https://d2l.ai/chapter_linear-regression/linear-regression-concise.html#sec-linear-concise), usamos una distribución normal para inicializar los valores de nuestros pesos. Si no especificamos el método de inicialización, el framework usará un método de inicialización aleatorio por defecto, que a menudo funciona bien en la práctica para tamaños de problemas moderados.

### Inicialización Xavier
<a id="subsec_xavier"></a>

Echemos un vistazo a la distribución a escala de una salida $o_{i}$ para alguna capa totalmente conectada *sin no linealidades*. Con $n_\textrm{in}$ entradas $x_j$ y sus pesos asociados $w_{ij}$ para esta capa, una salida es dada por

$$o_{i} = \sum_{j=1}^{n_\textrm{in}} w_{ij} x_j.$$

Los pesos $w_{ij}$ Además, supongamos que esta distribución tiene cero media y varianza. $\sigma^2$Tenga en cuenta que esto no significa que la distribución tiene que ser gaussiano, sólo que la media y la varianza necesitan existir. Por ahora, vamos a asumir que las entradas a la capa $x_j$ también tienen cero media y varianza $\gamma^2$ y que son independientes de $w_{ij}$ En este caso, podemos calcular la media de $o_i$:

$$
\begin{aligned}
    E[o_i] & = \sum_{j=1}^{n_\textrm{in}} E[w_{ij} x_j] \\&= \sum_{j=1}^{n_\textrm{in}} E[w_{ij}] E[x_j] \\&= 0, \end{aligned}$$

y la diferencia:

$$
\begin{aligned}
    \textrm{Var}[o_i] & = E[o_i^2] - (E[o_i])^2 \\
        & = \sum_{j=1}^{n_\textrm{in}} E[w^2_{ij} x^2_j] - 0 \\
        & = \sum_{j=1}^{n_\textrm{in}} E[w^2_{ij}] E[x^2_j] \\
        & = n_\textrm{in} \sigma^2 \gamma^2.
\end{aligned}
$$

Una forma de mantener la varianza fija es establecer $n_\textrm{in} \sigma^2 = 1$. Ahora consideremos la retropropagación. Allí nos enfrentamos a un problema similar, aunque los gradientes se propagan desde las capas más cercanas a la salida. Usando el mismo razonamiento que para la propagación hacia delante, vemos que la varianza de gradientes puede explotar a menos que $n_\textrm{out} \sigma^2 = 1$, donde $n_\textrm{out}$ es el número de salidas de esta capa. Esto nos deja en un dilema: no es posible satisfacer ambas condiciones simultáneamente. En lugar de ello, simplemente tratamos de satisfacer:

$$
\begin{aligned}
\frac{1}{2} (n_\textrm{in} + n_\textrm{out}) \sigma^2 = 1 \textrm{ or equivalently }
\sigma = \sqrt{\frac{2}{n_\textrm{in} + n_\textrm{out}}}.
\end{aligned}
$$

Este es el razonamiento subyacente a la inicialización *Xavier* ahora estándar y prácticamente beneficiosa, nombrada en honor al primer autor de sus creadores [Glorot.Bengio.2010](https://d2l.ai/chapter_references/zreferences.html). Típicamente, las muestras de inicialización Xavier pesan de una distribución gaussiana con media cero y varianza $\sigma^2 = \frac{2}{n_\textrm{in} + n_\textrm{out}}$. También podemos adaptar esto para elegir la varianza cuando el muestreo pesa de una distribución uniforme. Tenga en cuenta que la distribución uniforme $U(-a, a)$ tiene variación $\frac{a^2}{3}$. Enchufe $\frac{a^2}{3}$ en nuestra condición en $\sigma^2$ nos impulsa a inicializar de acuerdo a

$$U\left(-\sqrt{\frac{6}{n_\textrm{in} + n_\textrm{out}}}, \sqrt{\frac{6}{n_\textrm{in} + n_\textrm{out}}}\right).$$

Aunque la suposición de no existencia de no linealidades en el razonamiento matemático anterior puede ser fácilmente violada en redes neuronales, el método de inicialización Xavier resulta funcionar bien en la práctica.

### Más allá
El razonamiento anterior apenas rasca la superficie de los enfoques modernos a la inicialización de parámetros. Un biblioteca de aprendizaje profundo a menudo implementa más de una docena de heurísticas diferentes. Además, la inicialización de parámetros sigue siendo un área caliente de investigación fundamental en el aprendizaje profundo. Entre ellos se encuentran heurísticas especializadas para parámetros vinculados (compartidos), super-resolución, modelos de secuencia, y otras situaciones.
[Xiao.Bahri.Sohl-Dickstein.ea.2018](https://d2l.ai/chapter_references/zreferences.html) mostró la posibilidad de entrenar
Redes neuronales de 10.000 capas sin trucos arquitectónicos mediante el uso de un método de inicialización cuidadosamente diseñado.

Si el tema te interesa, te sugerimos una inmersión profunda en las ofertas de este módulo, leyendo los artículos que proponen y analizan cada heurística, y luego explorando las últimas publicaciones sobre el tema.Quizás te tropezarás o incluso inventarás una idea inteligente y contribuirás a la implementación de bibliotecas de aprendizaje profundo.

## Resumen
Los gradientes que desaparecen y explotan son problemas comunes en redes profundas. Se requiere un gran cuidado en la inicialización de parámetros para asegurar que los gradientes y parámetros permanezcan bien controlados. Se necesitan heurísticas de inicialización para asegurar que los gradientes iniciales no sean ni demasiado grandes ni demasiado pequeños. La inicialización aleatoria es clave para asegurar que la simetría se rompa antes de la optimización. La inicialización de Xavier sugiere que, para cada capa, la varianza de cualquier salida no se vea afectada por el número de entradas, y la varianza de cualquier gradiente no se vea afectada por el número de salidas.

## Ejercicios
1. ¿Puede diseñar otros casos en los que una red neuronal pueda mostrar simetría que necesita romperse, además de la simetría de permutación en las capas de un MLP?
1. ¿Podemos inicializar todos los parámetros de peso en regresión lineal o en regresión softmax al mismo valor?
1. Busca límites analíticos sobre los valores propios del producto de dos matrices. ¿Qué te dice esto acerca de asegurar que los gradientes estén bien condicionados?
1. Si sabemos que algunos términos divergen, ¿podemos arreglar esto después del hecho? Mira el papel sobre escala de velocidad adaptable por capas para la inspiración [You.Gitman.Ginsburg.2017](https://d2l.ai/chapter_references/zreferences.html).


[Debate del original](https://discuss.d2l.ai/t/104)
